# Experimento 1 — LightGBM sem balanceamento utilizando 4 variáveis

Esse experimento utiliza o algoritmo LightGBM (Light Gradient Boosting
Machine) aplicado ao dataset de qualidade hídrica, mantendo as mesmas
condições dos experimentos anteriores para permitir comparação direta entre os
algoritmos.

O LightGBM constrói modelos de forma sequencial, onde cada árvore é
treinada para corrigir os erros cometidos pela anterior. Sua estratégia
de crescimento leaf-wise permite capturar padrões mais complexos nos dados,
o que pode resultar em melhor separação entre as classes de qualidade
hídrica — especialmente as minoritárias `Crítica` e `Atenção`, que
apresentaram baixo desempenho no Experimento 1.

As variáveis utilizadas como entrada foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`

Enquanto isso, a variável alvo (`y`) utilizada para previsão foi:

- `conama_status`

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada utilizando `train_test_split` com `random_state=42`
e `stratify=y`, mantendo a proporção original das classes nos conjuntos de
treino e teste.

Nenhuma técnica de balanceamento foi aplicada, mantendo a distribuição
original das classes. O objetivo é isolar o efeito da troca de algoritmo —
qualquer diferença nos resultados em relação aos experimentos 1 ao 6 pode ser
atribuída exclusivamente ao comportamento do LightGBM.

Após o treinamento, o modelo foi avaliado utilizando:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusão

## Preparação do ambiente

In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
# esperado: (141399, 22)

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type"
]]

y = df["conama_status"]

In [ ]:
# parâmetros - divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
                # sem class_weight — igual ao Experimento 1
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi
realizada a avaliação das métricas no conjunto de treino, permitindo
comparar o comportamento entre treino e teste e identificar possíveis
sinais de overfitting.

In [ ]:
# Métricas de treino
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.7506961695205934
Train Precision:
0.6984026769886671
Train Recall:
0.7506961695205934
Train F1:
0.7029669800592149
Train Confusion Matrix:
[[ 1684  2265     3  3608]
 [  922  3820     5 16931]
 [  272   391    53   390]
 [  740  2672     2 79361]]


In [ ]:
# Métricas de teste
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.745049504950495

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.42      0.21      0.28      1890
         Boa       0.39      0.16      0.23      5419
     Crítica       0.17      0.01      0.02       277
   Excelente       0.79      0.96      0.87     20694

    accuracy                           0.75     28280
   macro avg       0.44      0.33      0.35     28280
weighted avg       0.68      0.75      0.70     28280


Confusion Matrix:
[[  390   621     1   878]
 [  260   894     7  4258]
 [   67   100     3   107]
 [  207   697     7 19783]]


## Resultados Obtidos — Experimento 1

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.745
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

```python
0.750
```

### Comparação com Experimento 1 do Random Forest

| Métrica            | Exp 1 — RF s/ bal | Exp 1 — LGBM s/ bal |
|--------------------|-------------------|----------------------|
| Accuracy treino    | 0.88              | 0.75                 |
| Accuracy teste     | 0.71              | 0.745                |
| Precision Atenção  | 0.32              | 0.42                 |
| Precision Boa      | 0.33              | 0.39                 |
| Precision Crítica  | 0.10              | 0.17                 |
| Precision Excelente| 0.80              | 0.79                 |
| Recall Atenção     | 0.20              | 0.21                 |
| Recall Boa         | 0.24              | 0.16                 |
| Recall Crítica     | 0.05              | 0.01                 |
| Recall Excelente   | 0.89              | 0.96                 |
| F1 Crítica         | 0.06              | 0.02                 |
| F1 Atenção         | 0.24              | 0.28                 |
| F1 Boa             | 0.28              | 0.23                 |
| F1 Excelente       | 0.84              | 0.87                 |
| Overfitting        | 0.17              | 0.005                |

### Conclusão

O LightGBM sem balanceamento apresentou acurácia de teste de 0.745,
ligeiramente superior ao Random Forest (0.71), e reduziu o overfitting
de 0.17 para 0.005 — indicando capacidade de generalização
significativamente maior.

A análise de precision e recall revela um comportamento mais detalhado
por classe. O LightGBM melhorou a precision em `Atenção` (0.32 → 0.42)
e `Crítica` (0.10 → 0.17), o que significa que, quando o modelo previu
essas classes, acertou com maior frequência do que o Random Forest.
No entanto, o recall de ambas caiu — `Atenção` de 0.20 para 0.21
(variação marginal) e `Crítica` de 0.05 para 0.01. Isso indica que o
modelo passou a prever essas classes com mais cautela, mas em pouquíssimas
ocasiões — resultando em alta precision e recall próximo de zero para
`Crítica`, caracterizando um modelo que raramente detecta situações
críticas, mas quando detecta, tende a estar correto.

Esse padrão fica evidente no F1: `Crítica` caiu de 0.06 para 0.02,
pois o ganho de precision não compensou a queda expressiva no recall.
`Boa` seguiu o mesmo caminho — precision subiu de 0.33 para 0.39, mas
recall caiu de 0.24 para 0.16, resultando em queda do F1 de 0.28 para
0.23.

Em `Excelente`, o comportamento foi oposto: precision praticamente
estável (0.80 → 0.79) e recall aumentando de 0.89 para 0.96, o que
elevou o F1 de 0.84 para 0.87. O modelo concentrou seu aprendizado
nessa classe, identificando quase todas as amostras `Excelente` do
teste, em detrimento das demais.

O Experimento 3 verificará se a adição de `class_weight="balanced"`
consegue elevar o recall das classes minoritárias sem comprometer a
capacidade de generalização observada neste experimento.

In [ ]:
# Salvamento das métricas
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

resultados = {
    "experimento": "exp_01_lightgbm_sem_balanceamento",
    "algoritmo": "LightGBM",
    "balanceamento": "nenhum",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas.")

Métricas salvas.
